<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 26 - MapReduce (Práctica): Patrones avanzados

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Implementar un índice invertido con MapReduce
- Resolver problemas Top-N de forma distribuida
- Simular un JOIN entre dos datasets con MapReduce
- Comparar rendimiento secuencial vs MapReduce paralelo

---

## 0. Framework MapReduce (reutilizado de la sesión 25)

Cargamos el framework que construimos en la sesión anterior.

In [ ]:
from collections import defaultdict
from multiprocessing import Pool, cpu_count
import time
import re
import random
import heapq

class MapReduceFramework:
    def __init__(self, mapper, reducer, combiner=None, num_workers=1):
        self.mapper = mapper
        self.reducer = reducer
        self.combiner = combiner
        self.num_workers = num_workers

    def _apply_mapper(self, item):
        key, value = item
        return self.mapper(key, value)

    def ejecutar(self, datos):
        inicio = time.time()

        if self.num_workers > 1:
            with Pool(self.num_workers) as pool:
                resultados_map = pool.map(self._apply_mapper, datos)
        else:
            resultados_map = [self._apply_mapper(d) for d in datos]

        pares = []
        for resultado in resultados_map:
            pares.extend(resultado)

        if self.combiner:
            agrupados_local = defaultdict(list)
            for k, v in pares:
                agrupados_local[k].append(v)
            pares = []
            for k, vs in agrupados_local.items():
                pares.extend(self.combiner(k, vs))

        agrupados = defaultdict(list)
        for k, v in pares:
            agrupados[k].append(v)

        resultados = {}
        for key, values in sorted(agrupados.items()):
            for k, v in self.reducer(key, values):
                resultados[k] = v

        tiempo_total = time.time() - inicio
        return resultados, tiempo_total

print(f"Framework MapReduce cargado. CPUs disponibles: {cpu_count()}")

---

## 1. Índice invertido

Un índice invertido mapea cada palabra a la lista de documentos donde aparece. Es la base de los motores de búsqueda.

In [ ]:
documentos = [
    ("hadoop.txt", "hadoop es un framework para procesamiento distribuido de grandes conjuntos de datos"),
    ("spark.txt", "spark es un motor de procesamiento de datos rápido y de propósito general para big data"),
    ("mapreduce.txt", "mapreduce es un modelo de programación para el procesamiento distribuido de grandes datos"),
    ("hdfs.txt", "hdfs es un sistema de archivos distribuido diseñado para almacenar grandes conjuntos de datos"),
    ("python.txt", "python es un lenguaje de programación popular para big data y procesamiento de datos"),
    ("hive.txt", "hive proporciona una interfaz sql para consultar datos almacenados en hadoop"),
    ("pig.txt", "pig es una plataforma para analizar grandes conjuntos de datos en hadoop"),
]

STOP_WORDS = {"es", "un", "una", "de", "el", "la", "los", "las", "para", "y", "en", "que", "por", "con"}

def indice_mapper(doc_id, contenido):
    palabras = re.findall(r'\b\w+\b', contenido.lower())
    emitidos = set()
    resultado = []
    for palabra in palabras:
        if palabra not in STOP_WORDS and palabra not in emitidos:
            resultado.append((palabra, doc_id))
            emitidos.add(palabra)
    return resultado

def indice_reducer(key, values):
    docs_unicos = sorted(set(values))
    return [(key, docs_unicos)]

mr_indice = MapReduceFramework(indice_mapper, indice_reducer)
indice, tiempo = mr_indice.ejecutar(documentos)

print(f"Índice invertido construido en {tiempo:.4f}s")
print(f"Términos indexados: {len(indice)}\n")

print(f"{'Término':<20} {'Docs':>4}  {'Documentos'}")
print("-" * 70)
for termino, docs in sorted(indice.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"{termino:<20} {len(docs):>4}  {docs}")

print("\n--- Búsqueda en el índice ---")
consultas = ["hadoop", "procesamiento", "big", "sql"]
for q in consultas:
    if q in indice:
        print(f"  '{q}' → encontrado en {len(indice[q])} documento(s): {indice[q]}")
    else:
        print(f"  '{q}' → no encontrado")

---

## 2. Top-N: productos más vendidos

Encontrar los N productos con más ventas de un dataset grande.

In [ ]:
random.seed(42)
productos_populares = ["Laptop", "Smartphone", "Tablet", "Auriculares", "Smartwatch"]
productos_normales = [f"Producto_{i}" for i in range(50)]
todos_productos = productos_populares * 20 + productos_normales * 3

ventas = []
for i in range(5000):
    producto = random.choice(todos_productos)
    cantidad = random.randint(1, 10)
    ventas.append((f"venta_{i}", f"{producto}|{cantidad}"))

print(f"Total registros de ventas: {len(ventas):,}")

def topn_mapper(key, value):
    producto, cantidad = value.split("|")
    return [(producto, int(cantidad))]

def topn_reducer(key, values):
    return [(key, sum(values))]

def topn_combiner(key, values):
    return [(key, sum(values))]

mr_topn = MapReduceFramework(topn_mapper, topn_reducer, combiner=topn_combiner)
totales, tiempo = mr_topn.ejecutar(ventas)

N = 10
top_n = heapq.nlargest(N, totales.items(), key=lambda x: x[1])

print(f"\nTop {N} productos más vendidos (calculado en {tiempo:.4f}s):")
print(f"{'Pos':>4} {'Producto':<20} {'Unidades':>10} {'Barra'}")
print("-" * 60)
max_val = top_n[0][1]
for i, (producto, total) in enumerate(top_n, 1):
    barra = "█" * int(total / max_val * 30)
    print(f"{i:>4} {producto:<20} {total:>10,}  {barra}")

---

## 3. Join entre dos datasets

Simulamos un reduce-side join entre una tabla de ventas y una tabla de productos.

In [ ]:
tabla_productos = [
    ("prod", "P001|Laptop Gaming|1200.00|Electrónica"),
    ("prod", "P002|Mouse Inalámbrico|25.50|Periféricos"),
    ("prod", "P003|Monitor 4K|450.00|Electrónica"),
    ("prod", "P004|Teclado Mecánico|85.00|Periféricos"),
    ("prod", "P005|Webcam HD|60.00|Periféricos"),
    ("prod", "P006|SSD 1TB|95.00|Almacenamiento"),
]

random.seed(42)
tabla_ventas = []
for i in range(50):
    prod_id = f"P{random.randint(1,6):03d}"
    cantidad = random.randint(1, 5)
    fecha = f"2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    tabla_ventas.append(("venta", f"{prod_id}|{cantidad}|{fecha}"))

datos_join = tabla_productos + tabla_ventas

print(f"Productos: {len(tabla_productos)} registros")
print(f"Ventas: {len(tabla_ventas)} registros")

def join_mapper(key, value):
    campos = value.split("|")
    if key == "prod":
        prod_id = campos[0]
        return [(prod_id, ("PROD", campos[1], float(campos[2]), campos[3]))]
    else:
        prod_id = campos[0]
        return [(prod_id, ("VENTA", int(campos[1]), campos[2]))]

def join_reducer(key, values):
    info_producto = None
    ventas_list = []

    for v in values:
        if v[0] == "PROD":
            info_producto = v
        elif v[0] == "VENTA":
            ventas_list.append(v)

    resultados = []
    if info_producto and ventas_list:
        nombre = info_producto[1]
        precio = info_producto[2]
        categoria = info_producto[3]
        total_unidades = sum(v[1] for v in ventas_list)
        total_ingresos = total_unidades * precio
        resultados.append((key, {
            'nombre': nombre,
            'categoria': categoria,
            'precio': precio,
            'num_ventas': len(ventas_list),
            'unidades': total_unidades,
            'ingresos': total_ingresos
        }))

    return resultados

mr_join = MapReduceFramework(join_mapper, join_reducer)
resultado_join, tiempo = mr_join.ejecutar(datos_join)

print(f"\nJoin completado en {tiempo:.4f}s\n")
print(f"{'ID':<6} {'Producto':<22} {'Categoría':<16} {'Ventas':>7} {'Uds':>5} {'Ingresos':>12}")
print("-" * 75)
for pid, info in sorted(resultado_join.items()):
    print(f"{pid:<6} {info['nombre']:<22} {info['categoria']:<16} {info['num_ventas']:>7} {info['unidades']:>5} {info['ingresos']:>10,.2f}€")

total_global = sum(info['ingresos'] for info in resultado_join.values())
print(f"\n{'TOTAL':>70} {total_global:>10,.2f}€")

---

## 4. Filtrado distribuido: grep sobre logs

Patrón de filtrado: buscar patrones en un gran volumen de logs.

In [ ]:
random.seed(99)
niveles = ["INFO"] * 70 + ["WARNING"] * 20 + ["ERROR"] * 8 + ["CRITICAL"] * 2
servicios = ["auth", "api", "db", "cache", "web"]
mensajes_error = [
    "Connection timeout",
    "Disk space low",
    "Memory exceeded",
    "Service unavailable",
    "Authentication failed",
]

logs = []
for i in range(10_000):
    nivel = random.choice(niveles)
    servicio = random.choice(servicios)
    if nivel in ("ERROR", "CRITICAL"):
        msg = random.choice(mensajes_error)
    else:
        msg = f"Operación normal #{random.randint(1,1000)}"
    logs.append((f"log_{i}", f"{nivel}|{servicio}|{msg}"))

print(f"Total de logs: {len(logs):,}")

def filter_mapper(key, value):
    nivel, servicio, mensaje = value.split("|")
    if nivel in ("ERROR", "CRITICAL"):
        return [(servicio, (nivel, mensaje))]
    return []

def filter_reducer(key, values):
    resumen = defaultdict(int)
    for nivel, msg in values:
        resumen[f"{nivel}: {msg}"] += 1
    return [(key, dict(resumen))]

mr_filter = MapReduceFramework(filter_mapper, filter_reducer)
errores, tiempo = mr_filter.ejecutar(logs)

print(f"\nAnálisis de errores completado en {tiempo:.4f}s\n")

for servicio, detalle in sorted(errores.items()):
    total = sum(detalle.values())
    print(f"Servicio: {servicio} ({total} errores)")
    for msg, count in sorted(detalle.items(), key=lambda x: -x[1]):
        barra = "█" * min(count, 30)
        print(f"  {msg:<35} {count:>4}  {barra}")
    print()

---

## 5. Comparación de rendimiento: secuencial vs paralelo

Medimos el speedup real al usar múltiples workers en nuestro framework MapReduce.

In [ ]:
import matplotlib.pyplot as plt

random.seed(42)
datos_grandes = []
palabras_vocab = ["big", "data", "hadoop", "spark", "mapreduce", "cluster",
                  "nodo", "bloque", "distribuido", "paralelo", "python",
                  "java", "scala", "hive", "pig", "hdfs", "yarn",
                  "procesamiento", "análisis", "resultado"]

for i in range(200):
    texto = " ".join(random.choices(palabras_vocab, k=500))
    datos_grandes.append((f"doc_{i}", texto))

print(f"Dataset: {len(datos_grandes)} documentos, ~{sum(len(v) for _,v in datos_grandes):,} caracteres\n")

def wc_mapper(key, value):
    palabras = re.findall(r'\b\w+\b', value.lower())
    return [(p, 1) for p in palabras]

def wc_reducer(key, values):
    return [(key, sum(values))]

def wc_combiner(key, values):
    return [(key, sum(values))]

configs = [1, 2, 4]
tiempos = []

print(f"{'Workers':>8} {'Tiempo':>10} {'Speedup':>10}")
print("-" * 30)

t_base = None
for w in configs:
    w_real = min(w, cpu_count())
    mr = MapReduceFramework(wc_mapper, wc_reducer, combiner=wc_combiner, num_workers=w_real)
    _, t = mr.ejecutar(datos_grandes)
    tiempos.append(t)
    if t_base is None:
        t_base = t
    speedup = t_base / t
    print(f"{w_real:>8} {t:>9.3f}s {speedup:>9.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar([str(c) for c in configs], tiempos, color=['#3498db', '#2ecc71', '#f39c12'])
axes[0].set_xlabel('Número de Workers')
axes[0].set_ylabel('Tiempo (segundos)')
axes[0].set_title('Tiempo de ejecución por configuración')

speedups = [t_base / t for t in tiempos]
ideal = [c for c in configs]
axes[1].plot(configs, speedups, 'bo-', linewidth=2, markersize=8, label='Speedup real')
axes[1].plot(configs, ideal, 'r--', linewidth=1, label='Speedup ideal')
axes[1].set_xlabel('Número de Workers')
axes[1].set_ylabel('Speedup')
axes[1].set_title('Speedup: real vs ideal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Data skew: el problema de las claves desbalanceadas

Demostración de cómo una clave muy popular puede desbalancear todo el procesamiento.

In [ ]:
import matplotlib.pyplot as plt

random.seed(42)

datos_equilibrados = []
for i in range(10_000):
    clave = f"key_{random.randint(0, 99)}"
    datos_equilibrados.append((f"r_{i}", f"{clave}|{random.randint(1,100)}"))

datos_sesgados = []
for i in range(10_000):
    if random.random() < 0.6:
        clave = "key_HOT"
    else:
        clave = f"key_{random.randint(0, 99)}"
    datos_sesgados.append((f"r_{i}", f"{clave}|{random.randint(1,100)}"))

def count_mapper(key, value):
    clave, valor = value.split("|")
    return [(clave, int(valor))]

def count_reducer(key, values):
    return [(key, sum(values))]

mr = MapReduceFramework(count_mapper, count_reducer)
res_eq, t_eq = mr.ejecutar(datos_equilibrados)
res_sk, t_sk = mr.ejecutar(datos_sesgados)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

vals_eq = sorted(res_eq.values(), reverse=True)[:20]
axes[0].bar(range(len(vals_eq)), vals_eq, color='#2ecc71')
axes[0].set_title(f'Distribución equilibrada (tiempo: {t_eq:.3f}s)', fontsize=12)
axes[0].set_xlabel('Claves (top 20)')
axes[0].set_ylabel('Suma de valores')

vals_sk = sorted(res_sk.values(), reverse=True)[:20]
colores_sk = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(vals_sk))]
axes[1].bar(range(len(vals_sk)), vals_sk, color=colores_sk)
axes[1].set_title(f'Distribución sesgada (tiempo: {t_sk:.3f}s)', fontsize=12)
axes[1].set_xlabel('Claves (top 20)')
axes[1].set_ylabel('Suma de valores')

plt.tight_layout()
plt.show()

max_key = max(res_sk.items(), key=lambda x: x[1])
total = sum(res_sk.values())
print(f"Clave más popular: '{max_key[0]}' con {max_key[1]:,} ({max_key[1]/total*100:.1f}% del total)")
print(f"\nEn un sistema real, el reducer de '{max_key[0]}' recibiría el 60% de los datos")
print("mientras los demás reducers terminan rápidamente → cuello de botella")
print("\nSolución: 'salting' → dividir key_HOT en key_HOT_0, key_HOT_1, key_HOT_2...")

---

## 7. Pipeline encadenado: ETL completo

Encadenamos varios jobs MapReduce para un pipeline ETL: limpiar → agregar → top-N.

In [ ]:
random.seed(42)
ciudades = ["Madrid", "Barcelona", "Sevilla", "Valencia", "Bilbao",
            "Málaga", "Zaragoza", "Palma", "Las Palmas", "Alicante"]
categorias = ["Hotel", "Restaurante", "Museo", "Transporte", "Compras"]

transacciones = []
for i in range(20_000):
    ciudad = random.choice(ciudades)
    cat = random.choice(categorias)
    importe = round(random.uniform(5, 500), 2)
    if random.random() < 0.02:
        importe = -abs(importe)
    transacciones.append((f"tx_{i}", f"{ciudad}|{cat}|{importe}"))

print(f"Transacciones generadas: {len(transacciones):,}")

print("\n--- JOB 1: Limpieza (filtrar importes negativos) ---")

def limpiar_mapper(key, value):
    ciudad, cat, importe = value.split("|")
    importe = float(importe)
    if importe > 0:
        return [(f"{ciudad}|{cat}", importe)]
    return []

def limpiar_reducer(key, values):
    return [(key, values)]

mr1 = MapReduceFramework(limpiar_mapper, limpiar_reducer)
datos_limpios, t1 = mr1.ejecutar(transacciones)

total_limpio = sum(len(v) for v in datos_limpios.values())
descartados = len(transacciones) - total_limpio
print(f"  Registros válidos: {total_limpio:,} (descartados: {descartados})")
print(f"  Tiempo: {t1:.4f}s")

print("\n--- JOB 2: Agregación por ciudad+categoría ---")

datos_para_agregar = []
for key, values in datos_limpios.items():
    for v in values:
        datos_para_agregar.append(("agg", f"{key}|{v}"))

def agregar_mapper(key, value):
    partes = value.split("|")
    ciudad_cat = f"{partes[0]}|{partes[1]}"
    importe = float(partes[2])
    return [(ciudad_cat, (importe, 1))]

def agregar_reducer(key, values):
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    return [(key, {'total': round(total, 2), 'count': count, 'media': round(total/count, 2)})]

def agregar_combiner(key, values):
    total = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    return [(key, (total, count))]

mr2 = MapReduceFramework(agregar_mapper, agregar_reducer, combiner=agregar_combiner)
agregados, t2 = mr2.ejecutar(datos_para_agregar)
print(f"  Grupos: {len(agregados)}")
print(f"  Tiempo: {t2:.4f}s")

print("\n--- JOB 3: Top 10 por gasto total ---")

top10 = heapq.nlargest(10, agregados.items(), key=lambda x: x[1]['total'])

print(f"\n{'Pos':>4} {'Ciudad':<15} {'Categoría':<15} {'Total':>12} {'Txs':>6} {'Media':>10}")
print("-" * 68)
for i, (key, info) in enumerate(top10, 1):
    ciudad, cat = key.split("|")
    print(f"{i:>4} {ciudad:<15} {cat:<15} {info['total']:>10,.2f}€ {info['count']:>6} {info['media']:>8,.2f}€")

print(f"\nPipeline completo: {t1+t2:.4f}s total")

---

## Recursos

- [MapReduce Design Patterns (O'Reilly)](https://www.oreilly.com/library/view/mapreduce-design-patterns/9781449341954/)
- [MapReduce Patterns, Algorithms and Use Cases](https://highlyscalable.wordpress.com/2012/02/01/mapreduce-patterns/)
- [mrjob - Python MapReduce](https://mrjob.readthedocs.io/)
- [Designing Data-Intensive Applications](https://dataintensive.net/)